In [1]:
import xarray as xr

dataset = xr.open_dataset(
    "data/raw/sci_mpsh-l2-avg5m_g16_d20250401_v2-0-2.nc"
)

print(dataset)

<xarray.Dataset> Size: 1MB
Dimensions:                           (time: 288, telescopes: 5,
                                       proton_diff_chans: 11,
                                       electron_diff_chans: 10)
Coordinates:
  * time                              (time) datetime64[ns] 2kB 2025-04-01 .....
Dimensions without coordinates: telescopes, proton_diff_chans,
                                electron_diff_chans
Data variables: (12/27)
    L1bRecordsInAvg                   (time) float64 2kB ...
    yaw_flip_flag                     (time) float32 1kB ...
    EclipseFlag                       (time) float64 2kB ...
    AvgDiffProtonFlux                 (time, telescopes, proton_diff_chans) float32 63kB ...
    AvgDiffProtonFluxUncert           (time, telescopes, proton_diff_chans) float32 63kB ...
    DiffProtonValidL1bSamplesInAvg    (time, telescopes, proton_diff_chans) float64 127kB ...
    ...                                ...
    IntDQFoobSum                      (time

In [4]:
for var in dataset.data_vars:

    print("="*80)

    print(var)

    print(dataset[var])

L1bRecordsInAvg
<xarray.DataArray 'L1bRecordsInAvg' (time: 288)> Size: 2kB
[288 values with dtype=float64]
Coordinates:
  * time     (time) datetime64[ns] 2kB 2025-04-01 ... 2025-04-01T23:55:00
Attributes:
    long_name:  Number of L1b 1-second records in average
    valid_min:  0
    valid_max:  301
yaw_flip_flag
<xarray.DataArray 'yaw_flip_flag' (time: 288)> Size: 1kB
[288 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 2kB 2025-04-01 ... 2025-04-01T23:55:00
Attributes:
    long_name:      Yaw flip flag, indicating if spacecraft is inverted
    comments:       While spacecraft is flipping, flag will be set to 1 or _F...
    valid_min:      0
    valid_max:      2
    flag_values:    [0 1 2]
    flag_meanings:  upright neither inverted
EclipseFlag
<xarray.DataArray 'EclipseFlag' (time: 288)> Size: 2kB
[288 values with dtype=float64]
Coordinates:
  * time     (time) datetime64[ns] 2kB 2025-04-01 ... 2025-04-01T23:55:00
Attributes:
    long_name:  Aggregate ec

In [6]:
import numpy as np
for var in dataset.data_vars:

    values = dataset[var].values

    missing = np.isnan(values).sum()

    total = values.size

    print(f"{var}")

    print(f"Missing : {missing}")

    print(f"Percentage : {100*missing/total:.3f}%")

    print()

L1bRecordsInAvg
Missing : 0
Percentage : 0.000%

yaw_flip_flag
Missing : 0
Percentage : 0.000%

EclipseFlag
Missing : 0
Percentage : 0.000%

AvgDiffProtonFlux
Missing : 0
Percentage : 0.000%

AvgDiffProtonFluxUncert
Missing : 55
Percentage : 0.347%

DiffProtonValidL1bSamplesInAvg
Missing : 0
Percentage : 0.000%

DiffProtonDQFdtcSum
Missing : 0
Percentage : 0.000%

DiffProtonDQFoobSum
Missing : 0
Percentage : 0.000%

DiffProtonDQFerrSum
Missing : 0
Percentage : 0.000%

AvgDiffElectronFlux
Missing : 0
Percentage : 0.000%

AvgDiffElectronFluxUncert
Missing : 50
Percentage : 0.347%

DiffElectronEffectiveEnergy
Missing : 0
Percentage : 0.000%

DiffElectronValidL1bSamplesInAvg
Missing : 0
Percentage : 0.000%

DiffElectronDQFdtcSum
Missing : 0
Percentage : 0.000%

DiffElectronDQFoobSum
Missing : 0
Percentage : 0.000%

DiffElectronDQFerrSum
Missing : 0
Percentage : 0.000%

AvgIntElectronFlux
Missing : 0
Percentage : 0.000%

AvgIntElectronFluxUncert
Missing : 5
Percentage : 0.347%

IntElectronE

In [8]:
df = dataset.to_dataframe()

print(df.head())

print(df.columns)

                                                             L1bRecordsInAvg  \
time       telescopes proton_diff_chans electron_diff_chans                    
2025-04-01 0          0                 0                              300.0   
                                        1                              300.0   
                                        2                              300.0   
                                        3                              300.0   
                                        4                              300.0   

                                                             yaw_flip_flag  \
time       telescopes proton_diff_chans electron_diff_chans                  
2025-04-01 0          0                 0                              0.0   
                                        1                              0.0   
                                        2                              0.0   
                                        3        

In [9]:
from pathlib import Path
import xarray as xr
import pandas as pd
from tqdm import tqdm


class GOESReader:

    def __init__(self, root_folder):

        self.root = Path(root_folder)

    def get_all_files(self):

        return sorted(self.root.rglob("*.nc"))

    def read_single_file(self, filepath):

        ds = xr.open_dataset(filepath)

        df = ds.to_dataframe().reset_index()

        return df

    def read_all(self):

        files = self.get_all_files()

        print(f"Found {len(files)} GOES files")

        all_data = []

        for file in tqdm(files):

            try:

                df = self.read_single_file(file)

                all_data.append(df)

            except Exception as e:

                print(file)

                print(e)

        return pd.concat(all_data, ignore_index=True)

In [10]:
reader = GOESReader(
    "data/raw/GOES"
)

df = reader.read_all()

print(df.head())

print(df.shape)

Found 1811 GOES files


  2%|▏         | 39/1811 [00:56<43:43,  1.48s/it]

data\raw\GOES\2020\02\sci_mpsh-l2-avg5m_g16_d20200210_v1-0-2.nc
[Errno -101] NetCDF: HDF error: 'c:\\Users\\sajal\\OneDrive\\Desktop\\BAH-ps14\\data\\raw\\GOES\\2020\\02\\sci_mpsh-l2-avg5m_g16_d20200210_v1-0-2.nc'


  7%|▋         | 120/1811 [02:55<45:03,  1.60s/it]

data\raw\GOES\2020\05\sci_mpsh-l2-avg5m_g16_d20200502_v1-0-2.nc
[Errno -101] NetCDF: HDF error: 'c:\\Users\\sajal\\OneDrive\\Desktop\\BAH-ps14\\data\\raw\\GOES\\2020\\05\\sci_mpsh-l2-avg5m_g16_d20200502_v1-0-2.nc'


...

 87%|████████▋ | 1572/1811 [1:12:03<11:12,  2.81s/it]

data\raw\GOES\2024\05\sci_mpsh-l2-avg5m_g16_d20240506_v2-0-2.nc



 87%|████████▋ | 1573/1811 [1:12:07<12:01,  3.03s/it]

data\raw\GOES\2024\05\sci_mpsh-l2-avg5m_g16_d20240507_v2-0-2.nc
Unable to allocate 1.21 MiB for an array with shape (288, 5, 11, 10) and data type float64


: 

In [1]:
import xarray as xr

ds = xr.open_dataset(
        "data/raw/sci_mpsh-l2-avg5m_g16_d20250401_v2-0-2.nc"
)

for var in [
    "AvgDiffElectronFlux",
    "AvgDiffProtonFlux",
    "AvgIntElectronFlux",
    "EclipseFlag",
    "yaw_flip_flag"
]:
    if var in ds:
        print(var, ds[var].shape, ds[var].dims)

AvgDiffElectronFlux (288, 5, 10) ('time', 'telescopes', 'electron_diff_chans')
AvgDiffProtonFlux (288, 5, 11) ('time', 'telescopes', 'proton_diff_chans')
AvgIntElectronFlux (288, 5) ('time', 'telescopes')
EclipseFlag (288,) ('time',)
yaw_flip_flag (288,) ('time',)


In [1]:
from cdflib import CDF

cdf = CDF("data/raw/WIND/2024/omni_hro_5min_20241201_v01.cdf")

info = cdf.cdf_info()

print("="*80)
print(info)

print("\nVariables\n")

for var in info.zVariables:
    print(var)

CDFInfo(CDF=WindowsPath('C:/Users/sajal/OneDrive/Desktop/BAH-ps14/data/raw/WIND/2024/omni_hro_5min_20241201_v01.cdf'), Version='3.9.1', Encoding=1, Majority='Row_major', rVariables=[], zVariables=['Epoch', 'YR', 'Day', 'HR', 'Minute', 'IMF', 'PLS', 'IMF_PTS', 'PLS_PTS', 'percent_interp', 'Timeshift', 'RMS_Timeshift', 'Time_btwn_obs', 'F', 'BX_GSE', 'BY_GSE', 'BZ_GSE', 'BY_GSM', 'BZ_GSM', 'RMS_SD_B', 'RMS_SD_fld_vec', 'flow_speed', 'Vx', 'Vy', 'Vz', 'proton_density', 'T', 'Pressure', 'E', 'Beta', 'Mach_num', 'Mgs_mach_num', 'x', 'y', 'z', 'BSN_x', 'BSN_y', 'BSN_z', 'AE_INDEX', 'AL_INDEX', 'AU_INDEX', 'SYM_D', 'SYM_H', 'ASY_D', 'ASY_H', 'PC_N_INDEX', 'PR-FLX_10', 'PR-FLX_30', 'PR-FLX_60'], Attributes=[{'Project': 'Global'}, {'Discipline': 'Global'}, {'Source_name': 'Global'}, {'Data_type': 'Global'}, {'Descriptor': 'Global'}, {'Data_version': 'Global'}, {'TITLE': 'Global'}, {'TEXT': 'Global'}, {'MODS': 'Global'}, {'Logical_file_id': 'Global'}, {'PI_name': 'Global'}, {'PI_affiliation': 'G